In [15]:
import numpy as np
from matplotlib import pyplot as plt
from analysis import Analysis
from stngpe import STN_GPe_loop
from utils import load_yaml, save_yaml
import scipy
from numpy.fft import fft, ifft
import math
from dbs import *
from tqdm.autonotebook import tqdm
import os
import torch
import pickle
from collections import defaultdict

In [16]:
DIR_NAMES = ['PD']

In [17]:
SYNC_PD = defaultdict(dict)
ENTROPY_PD = defaultdict(dict)
STD_DEV_PD = defaultdict(dict)
STD_DEV_Processed_PD = defaultdict(dict)
POWER_BETA_PD = defaultdict(dict)
POWER_BETA_PD_1 = defaultdict(dict)
POWER_BETA_PD_2 = defaultdict(dict)

In [ ]:
for dir_name in tqdm(DIR_NAMES):
    with open(f'./PD/{dir_name}/spikes.pkl', 'rb') as f:
        SPIKES_DATA = pickle.load(f)
    with open(f'./PD/{dir_name}/lfp.pkl', 'rb') as f:
        LFP_DATA = pickle.load(f)
    lat_strength_arr = np.array(list(SPIKES_DATA.keys()))

    print(f'*******************************{dir_name}*******************************')
    for i in tqdm(range(len(lat_strength_arr))):

        stn_lat = lat_strength_arr[i]
        Analysis_PD= Analysis(SPIKES_DATA[stn_lat])

        # ******************* Power spectrum analysis ********************
        spectral_entropy_PD = Analysis_PD.spectral_entropy(LFP_DATA[stn_lat], fs = 10000, nperseg=10000, fmax = 50)

        print(f'lat: {stn_lat}; Entropy: {spectral_entropy_PD}')

  0%|          | 0/1 [00:00<?, ?it/s]

*******************************PD*******************************


  0%|          | 0/10 [00:00<?, ?it/s]

lat: 0.0275; Entropy: 0.3076661812932494
lat: 0.03; Entropy: 0.3294829950348941
lat: 0.0325; Entropy: 0.30913679485158674
lat: 0.035; Entropy: 0.28158443330149585
lat: 0.037500000000000006; Entropy: 0.2863023114695961
lat: 0.04; Entropy: 0.3338766956323873
lat: 0.0425; Entropy: 0.34816079904685987
lat: 0.045; Entropy: 0.2906748969677418
lat: 0.0475; Entropy: 0.30131718143697156
lat: 0.05; Entropy: 0.35598170436637994


In [20]:
for dir_name in tqdm(DIR_NAMES):
    with open(f'./PD/{dir_name}/spikes.pkl', 'rb') as f:
        SPIKES_DATA = pickle.load(f)
    with open(f'./PD/{dir_name}/lfp.pkl', 'rb') as f:
        LFP_DATA = pickle.load(f)
    lat_strength_arr = np.array(list(SPIKES_DATA.keys()))

    print(f'*******************************{dir_name}*******************************')
    for i in tqdm(range(len(lat_strength_arr))):

        stn_lat = lat_strength_arr[i]
        Analysis_PD= Analysis(SPIKES_DATA[stn_lat])

        # ******************* Synchrony analysis **********************
        Rvalue_PD, Ravg_PD = Analysis_PD.synchrony()

        # ******************* Power spectrum analysis ********************
        spectral_entropy_PD = Analysis_PD.spectral_entropy(LFP_DATA[stn_lat], fs = 10000, nperseg=10000, fmax = 50)


        #******************* Power in beta ****************************
        power_peak_1, power_peak_2, power_beta = Analysis_PD.power_beta(LFP_DATA[stn_lat], fs = 10000)

        # ******************* mean rate *********************************
        rate_data_PD = Analysis_PD.spike_rate(binsize = 100)
        STN_4_PD = torch.tensor(np.array([rate_data_PD['1'], rate_data_PD['2'], rate_data_PD['3'], rate_data_PD['4']]))
        STN_4_PD_processed = torch.mean(STN_4_PD.reshape(-1,200,100), dim = 2)
        STN_4_PD_std = torch.std(STN_4_PD, dim = 0)
        STN_4_PD_std_processed = torch.std(STN_4_PD_processed, dim = 0)


        # ******************* Storing data in a dictionary *****************
        SYNC_PD[dir_name][stn_lat] = Ravg_PD
        ENTROPY_PD[dir_name][stn_lat] = spectral_entropy_PD
        STD_DEV_PD[dir_name][stn_lat] = STN_4_PD_std
        STD_DEV_Processed_PD[dir_name][stn_lat] = STN_4_PD_std_processed
        POWER_BETA_PD[dir_name][stn_lat] = power_beta
        POWER_BETA_PD_1[dir_name][stn_lat] = power_peak_1
        POWER_BETA_PD_2[dir_name][stn_lat] = power_peak_2

        print(f'lat: {stn_lat}; Synchrony: {Ravg_PD}; Entropy: {spectral_entropy_PD}; STD: {STN_4_PD_std.mean()}; Processed STD: {STN_4_PD_std_processed.mean()}, Power in beta: {power_beta}')


  0%|          | 0/1 [00:00<?, ?it/s]

*******************************PD*******************************


  0%|          | 0/10 [00:00<?, ?it/s]

lat: 0.0275; Synchrony: 0.9162673731037164; Entropy: 0.3076661812932494; STD: 0.009806579575486802; Processed STD: 0.0031963402060171814, Power in beta: 97.97999908423391
lat: 0.03; Synchrony: 0.976391740018015; Entropy: 0.3294829950348941; STD: 0.005175784059456877; Processed STD: 0.002049990290659674, Power in beta: 99.05036161807887
lat: 0.0325; Synchrony: 0.880625473653735; Entropy: 0.30913679485158674; STD: 0.017795967598406324; Processed STD: 0.009504309759377385, Power in beta: 99.08300209435066
lat: 0.035; Synchrony: 0.6855289255374934; Entropy: 0.28158443330149585; STD: 0.02094886667036592; Processed STD: 0.012697050558988033, Power in beta: 99.60491823676229
lat: 0.037500000000000006; Synchrony: 0.7787836915267883; Entropy: 0.2863023114695961; STD: 0.024850021967140097; Processed STD: 0.012047747435127147, Power in beta: 85.56844164871887
lat: 0.04; Synchrony: 0.8577557656399164; Entropy: 0.3338766956323873; STD: 0.014485499431770939; Processed STD: 0.006470284932080715, Powe

In [21]:
DATA = {
    'SYNC_PD': SYNC_PD,
    'ENTROPY_PD': ENTROPY_PD,
    'STD_DEV_PD': STD_DEV_PD,
    'STD_DEV_Processed_PD': STD_DEV_Processed_PD,
    'POWER_BETA_PD': POWER_BETA_PD,
    'POWER_BETA_PD_1': POWER_BETA_PD_1,
    'POWER_BETA_PD_2': POWER_BETA_PD_2
}

with open(f'./PD/ANALYSIS_DATA.pkl', 'wb') as f:
    pickle.dump(DATA, f)